# CEOAI Practice 1 - Stochastic Rift Minimum Solution

Objective: build the smallest end-to-end value-estimation pipeline:

1. Load transition logs and query states.
2. Estimate an empirical MDP.
3. Run value iteration.
4. Export `predictions.csv`.

This notebook uses the local fixture data in `data/`. Replace that folder with the official Nitro files before judging.

In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
GAMMA = 0.99

In [ ]:
logs = pd.read_csv(DATA / "sector_logs.csv")
queries = pd.read_csv(DATA / "query_states.csv")

spec = importlib.util.spec_from_file_location("env", DATA / "env.py")
env_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(env_module)
env = env_module.Sector7Env(seed=0)

n_states = int(env.n_states)
n_actions = int(env.n_actions)
print(logs.head())
print({"rows": len(logs), "states": n_states, "actions": n_actions, "queries": len(queries)})

   current_state  action   reward  next_state
0             13       3  -2.4508          12
1             22       3  30.4714          23
2              9       3  -4.5564          13
3             18       0  -1.0307          19
4             13       2  -0.7471          16
{'rows': 6000, 'states': 24, 'actions': 4, 'queries': 12}


In [ ]:
grouped = (
    logs.groupby(["current_state", "action", "next_state"], as_index=False)
    .agg(count=("reward", "size"), reward=("reward", "mean"))
)

model = {}
global_reward = float(logs["reward"].mean())
for state in range(n_states):
    for action in range(n_actions):
        block = grouped[(grouped.current_state == state) & (grouped.action == action)]
        if len(block) == 0:
            # Conservative fallback for unseen pairs.
            model[(state, action)] = [(1.0, global_reward - 1.0, state)]
            continue
        total = block["count"].sum()
        model[(state, action)] = [
            (row["count"] / total, row["reward"], int(row["next_state"]))
            for _, row in block.iterrows()
        ]

known_pairs = logs[["current_state", "action"]].drop_duplicates().shape[0]
print({"known_state_action_pairs": known_pairs, "total_pairs": n_states * n_actions})

{'known_state_action_pairs': 96, 'total_pairs': 96}


In [ ]:
values = np.zeros(n_states, dtype=float)
deltas = []
for iteration in range(2500):
    old = values.copy()
    for state in range(n_states):
        q_values = []
        for action in range(n_actions):
            q = sum(prob * (reward + GAMMA * old[next_state]) for prob, reward, next_state in model[(state, action)])
            q_values.append(q)
        values[state] = max(q_values)
    delta = float(np.max(np.abs(values - old)))
    deltas.append(delta)
    if delta < 1e-6:
        break

print({"iterations": len(deltas), "last_delta": deltas[-1]})

{'iterations': 812, 'last_delta': 9.933948206253262e-07}


In [ ]:
pred = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": queries["id"],
    "answer": queries["state_id"].map(lambda s: float(values[int(s)])),
})
pred.to_csv(OUT / "predictions.csv", index=False)
pred.head()

,subtaskID,datapointID,answer
0,1,0,31.634493
1,1,1,33.478420
2,1,2,33.498980
3,1,3,35.573188
4,1,4,36.537246


In [ ]:
truth_path = DATA / "ground_truth_values.csv"
if truth_path.exists():
    truth = pd.read_csv(truth_path)
    mse = np.mean((pred["answer"].to_numpy() - truth["true_value"].to_numpy()) ** 2)
    print({"fixture_mse": float(mse)})

assert len(pred) == len(queries)
assert list(pred.columns) == ["subtaskID", "datapointID", "answer"]
assert pred["answer"].notna().all()
print("wrote", OUT / "predictions.csv")

{'fixture_mse': 5.580676848096618}
wrote d:\projects\Supervised-Learning-Experiments\olympiads\competition_samples\raw\ceoai-2026-practice-rounds\round-1\stochastic_rift\outputs\predictions.csv
